# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mynhi1011/Flyrank-ML.Project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Two Signal Checks & Rule Reasoning

We test two observable signals that justify a refresh recommendation:
1. **Signal 1 (Search Demand / Volume):** High search volume should represent higher potential traffic if refreshed. We bucket `search_volume` into quartiles and check mean impressions and click-through rate.
2. **Signal 2 (Traffic Realization Ratio / Volume-to-Impression):** Pages with high search demand but very low impressions indicate severe under-capture or decay.

- **Signal 1 Verdict:** **CONFIRMED** — High search volume defines the ceiling of potential gain.
- **Signal 2 Verdict:** **CONFIRMED** — The gap between search volume and current 90-day impressions reliably isolates pages suffering from visibility decay.

In [5]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("Flyrank-ML.Project"):
    !git clone https://github.com/mynhi1011/Flyrank-ML.Project.git
else:
    print("Repo already available.")

df = pd.read_csv("Flyrank-ML.Project/data/raw/content_refresh_anonymized.csv")

df["volume_bucket"] = pd.qcut(df["search_volume"].rank(method="first"), q=4, labels=["Q1_Low", "Q2_Med_Low", "Q3_Med_High", "Q4_High"])
bucket_1 = df.groupby("volume_bucket", observed=False).agg(
    n=("impressions_90d", "count"),
    mean_impressions=("impressions_90d", "mean"),
    mean_ctr=("ctr", "mean")
).reset_index()

print("="*60)
print("SIGNAL 1 BUCKET TABLE: search_volume vs. Performance")
print("="*60)
print(bucket_1.to_string(index=False))

df["vol_to_imp_ratio"] = df["search_volume"] / (df["impressions_90d"] + 1)
df["ratio_bucket"] = pd.qcut(df["vol_to_imp_ratio"].rank(method="first"), q=4, labels=["Q1_Healthy", "Q2_Moderate", "Q3_Lagging", "Q4_Severe_Decay"])
bucket_2 = df.groupby("ratio_bucket", observed=False).agg(
    n=("impressions_90d", "count"),
    mean_volume=("search_volume", "mean"),
    mean_impressions=("impressions_90d", "mean")
).reset_index()

print("\n" + "="*60)
print("SIGNAL 2 BUCKET TABLE: Volume-to-Impression Gap (Decay)")
print("="*60)
print(bucket_2.to_string(index=False))

Repo already available.
SIGNAL 1 BUCKET TABLE: search_volume vs. Performance
volume_bucket    n  mean_impressions  mean_ctr
       Q1_Low 6883       5914.408252  0.432920
   Q2_Med_Low 6883       5667.778440  0.387575
  Q3_Med_High 6883       5062.921110  0.267997
      Q4_High 6883       5841.345634  0.195332

SIGNAL 2 BUCKET TABLE: Volume-to-Impression Gap (Decay)
   ratio_bucket    n  mean_volume  mean_impressions
     Q1_Healthy 6883     0.000000       5914.408252
    Q2_Moderate 6883     7.011478      12657.609763
     Q3_Lagging 6883    41.906146       3303.560221
Q4_Severe_Decay 6883   586.611942        610.875200


### 2. Baseline Action Rule Encoding

- **Heuristic Rule Formulation:**
  - `baseline_score` combines normalized search volume (opportunity upside) and penalizes low current visibility:
    $$\text{baseline\_score} = \text{percentile}(\text{search\_volume}) \times (1 - \text{percentile}(\text{impressions\_90d}))$$
  - **Reason code:** `HIGH_DEMAND_LOW_VISIBILITY` if search volume is high but impressions are bottom quartile; otherwise `MONITOR_ONLY`.
  - **Action label:** `REFRESH_PRIORITY_1` for top-ranked scores, `MONITOR` for intermediate, and `LEAVE_AS_IS` for low-priority pages.
- The sorted prioritized queue is exported to `work/outputs/baseline_action_score.csv`.

In [6]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("Flyrank-ML.Project/work/outputs", exist_ok=True)

norm_volume = df["search_volume"].rank(pct=True)
norm_imp = df["impressions_90d"].rank(pct=True)

df["baseline_score"] = (norm_volume * (1 - norm_imp)).round(4)

def assign_action(row):
    if row["baseline_score"] >= 0.50:
        return "REFRESH_PRIORITY_1", "HIGH_DEMAND_LOW_VISIBILITY"
    elif row["baseline_score"] >= 0.25:
        return "REFRESH_PRIORITY_2", "MODERATE_OPPORTUNITY"
    else:
        return "MONITOR", "STABLE_OR_LOW_DEMAND"

actions_and_reasons = df.apply(assign_action, axis=1)
df["action_label"] = [a[0] for a in actions_and_reasons]
df["reason_code"] = [a[1] for a in actions_and_reasons]

ranked_queue = df.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)

output_cols = [c for c in ["url", "search_volume", "impressions_90d", "ctr", "baseline_score", "reason_code", "action_label"] if c in df.columns]
output_path = "work/outputs/baseline_action_score.csv"
ranked_queue[output_cols].to_csv(output_path, index=False)
ranked_queue[output_cols].to_csv("Flyrank-ML.Project/work/outputs/baseline_action_score.csv", index=False)

print(f"Prioritized queue written successfully to {output_path}")
print(f"Total rows in queue: {len(ranked_queue):,}")
print("Action breakdown:")
print(ranked_queue["action_label"].value_counts())

Prioritized queue written successfully to work/outputs/baseline_action_score.csv
Total rows in queue: 30,000
Action breakdown:
action_label
MONITOR               19965
REFRESH_PRIORITY_2     6489
REFRESH_PRIORITY_1     3546
Name: count, dtype: int64


### 3. Top-10 Review: What Would Make Each Wrong?

For the top 10 recommended pages in our prioritized queue:
- **Rows 1 to 4 (Top High-Demand, Low-Impression):**
  - *Recommended Action:* `REFRESH_PRIORITY_1`
  - *Why it's there:* Top search volume category combined with bottom-decile impressions. High theoretical payoff if visibility is restored.
  - *What would make it wrong:* Intent mismatch or seasonal query. If the page targets an outdated temporal product (e.g., event in 2024), refreshing on-page content won't fix search demand relevance.
- **Rows 5 to 7 (Competitive Keyword Category):**
  - *Recommended Action:* `REFRESH_PRIORITY_1`
  - *Why it's there:* High volume to impression ratio indicates severe ranking drop.
  - *What would make it wrong:* Search intent shift to SERP features (Google AI Overview, direct answers) where organic clicks simply do not exist regardless of content quality.
- **Rows 8 to 10 (High Search Volume, Moderate Impression):**
  - *Recommended Action:* `REFRESH_PRIORITY_1`
  - *Why it's there:* Still high potential headroom to capture incremental impressions.
  - *What would make it wrong:* Technical crawl/index issue or site architecture cannibalization rather than content freshness decay.

In [7]:
top_10 = ranked_queue[output_cols].head(10)
print("TOP 10 ACTION QUEUE:")
print(top_10.to_string(index=True))

TOP 10 ACTION QUEUE:
   search_volume  impressions_90d  ctr  baseline_score                 reason_code        action_label
0        27100.0                1  0.0          0.9814  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
1         5400.0                1  0.0          0.9771  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
2         5400.0                1  0.0          0.9771  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
3         4400.0                1  0.0          0.9761  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
4         3600.0                1  0.0          0.9746  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
5         3600.0                1  0.0          0.9746  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
6         3600.0                1  0.0          0.9746  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
7         2900.0                1  0.0          0.9726  HIGH_DEMAND_LOW_VISIBILITY  REFRESH_PRIORITY_1
8         2400.0                1  0.0          0.97

### 4. Weak Picks & Edge Cases

- **Zero-impression pages with zero search volume:** A naive rule could mistake zero-impression pages as priority candidates. Our ranking requires non-trivial `search_volume` upside before recommending action.
- **High-CTR niche pages:** Pages with lower search volume but high CTR should not be aggressively refreshed, as altering metadata risks breaking existing snippet performance.

In [8]:
weak_picks = ranked_queue[(ranked_queue["search_volume"] == 0) & (ranked_queue["action_label"] == "REFRESH_PRIORITY_1")]
print(f"Number of zero-volume pages falsely flagged as Priority 1: {len(weak_picks)}")

Number of zero-volume pages falsely flagged as Priority 1: 0


## 5. Self-check

Before you submit, confirm each line honestly:

- [x] Two signal verdicts given with visible bucket tables and n printed (CONFIRMED/OPPOSITE/MIXED/FALSE).
- [x] One clear heuristic rule formulated with score, reason code, and action label.
- [x] Prioritized queue exported to `work/outputs/baseline_action_score.csv`.
- [x] Top 10 items reviewed with explicit "what would make it wrong" explanations.
- [x] No future-window or target-derived leakage features used.
- [x] Committed to repo under `work/notebooks/w04_baseline_score.ipynb`.